In [3]:
from pathlib import Path
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.colors import BoundaryNorm

# ---- Global variables ----
GENES = ['CDH5', 'MMP1', 'COL1A1', 'KDR', 'VWF', 'PECAM1', 'VEGFA', 'DLL4', 'ITGA6', 'SEMA3F', 'JAG1', 'FLT1', 'PDGFB', 'COL1A2', 'APLN', 'PDGFRB', 'ANGPT2', 'ICAM1']
IMAGE_KEY = "source_folder"          # one physical image per source_folder
output_dir = Path(".")
fig_dir = output_dir / "report_figures"
fig_dir.mkdir(exist_ok=True)

# ==========================================================================
# Read the saved outputs of parts 2-4 (this notebook only presents & synthesises them).
# ==========================================================================

# ---- part 2: pairwise gene co-occurrence (Fisher's exact, all cells) ----
pair_sig = pd.read_csv(output_dir / "all_cells_sig_pairs.csv", index_col=0)

# ---- part 3: gene-state clustering ----
cells = pd.read_csv(output_dir / "clustered_data_with_states.csv")           # per-cell (bmm, bmm_confidence, lpca1/2, group)
z_all = pd.read_csv(output_dir / "cluster_colocation_zscores.csv", index_col=0)   # cluster co-location z
cluster_overview = pd.read_csv(output_dir / "section2_cluster_overview.csv")      # per-cluster sizes / states
endo_dev = pd.read_csv(output_dir / "endo_lpca_deviance_explained.csv", index_col=0)["pct_deviance"]

# Map each Bernoulli cluster id -> its readable name (from the saved per-cell "group").
cluster_name_of = cells[cells["bmm"] >= 0].groupby("bmm")["group"].first().to_dict()

# ---- part 4: neighbourhood niches + gene-interaction modules ----
niche_gene_z = pd.read_csv(output_dir / "part4_niche_fingerprint.csv", index_col=0)          # niche x gene (z nbhd expr)
state_comp = pd.read_csv(output_dir / "part4_niche_state_composition.csv", index_col=0)      # niche x gene-state fraction
niche_module = pd.read_csv(output_dir / "part4_module_activity_per_niche.csv", index_col=0)  # niche x module activity
gene_modules = pd.read_csv(output_dir / "part4_gene_modules.csv")                            # gene, module, on_freq
module_edges = pd.read_csv(output_dir / "part4_module_edges.csv")                            # g1, g2, log2_OR

print("Loaded part 2/3/4 outputs:")
print(f"  part2  pairwise significant pairs : {len(pair_sig):,}")
print(f"  part3  cells                      : {len(cells):,}  |  gene-state clusters: {len(cluster_name_of)}")
print(f"  part4  niches                     : {niche_gene_z.shape[0]}  |  gene modules: {gene_modules['module'].nunique()}")


Loaded part 2/3/4 outputs:
  part2  pairwise significant pairs : 123
  part3  cells                      : 13,047  |  gene-state clusters: 6
  part4  niches                     : 6  |  gene modules: 5


### Conclusions - gene modules

**Approach.** A gene-gene graph from significant **positive** co-occurrences in endothelial
cells. Edge weight = **log2 odds ratio** (effect size). The **pan-endothelial markers** (`CDH5`,
`VWF`, `PECAM1`) and **collagen** (`COL1A1`, `COL1A2`) are **excluded** - they define the lineage /
stroma and carry no module information. Communities on the remaining functional genes = co-firing
**modules** (greedy modularity, resolution 1.5); module activity is then averaged per niche.

**How real are the modules? (quantified, not eyeballed).** Weighting edges by the **log2 odds
ratio** and taking the **resolution-1.5** greedy partition gives **5 modules (Q = 0.21)**. The two
angiogenic modules (VEGF-core, tip/sprouting) are metric- and resolution-stable; using log2 OR
(rather than phi) additionally splits the perivascular genes into a **mural** (`PDGFB`/`PDGFRB`)
and a **remodelling** (`ANGPT2`/`APLN`) module - phi keeps them merged as one - a split that is
independently favoured by the spatial niches (see below).

**Assumptions / drawbacks.** Co-occurrence is association, not causal regulation. Only positive
edges are used; the number of modules depends on the edge weight and resolution (phi -> 4, log2 OR
-> 5); and this is built on endothelial cells only (the dominant lineage).

**Biology - five co-firing modules:**
- **VEGF / angiogenic core** - `KDR`, `VEGFA`, `ITGA6`: the VEGF-activation programme.
- **Tip / sprouting** - `DLL4`, `MMP1`, `SEMA3F`, `FLT1`: Notch tip-selection + ECM-remodelling
  / guidance genes.
- **Mural / pericyte** - `PDGFB`, `PDGFRB`: pericyte recruitment (`PDGFB` ligand + `PDGFRB` receptor).
- **Remodelling / apelin** - `ANGPT2`, `APLN`: vessel destabilisation + apelin-tip remodelling.
- **Activated / inflammatory-Notch** - `ICAM1`, `JAG1`: endothelial activation / stalk-Notch.

**Do the methods agree?** The two angiogenic modules map cleanly onto the tissue: **VEGF-core and
tip/sprouting both light up the angiogenic niches** (the `MMP1`/`VEGFA` and `FLT1`/`DLL4` niches)
and match the `KDR`/`VEGFA` and `FLT1`/`DLL4` gene-state clusters. The smaller modules map to
distinct micro-environments too: **mural** (`PDGFB`/`PDGFRB`) -> the `PDGFRB` mural niche, and
**remodelling** (`ANGPT2`/`APLN`) -> the `APLN`/`ANGPT2` niche - so splitting the perivascular block
into mural vs remodelling is corroborated spatially. Only the **`ICAM1`/`JAG1`** pair stays a
gene-level association whose two genes sit in different niches, i.e. not spatially co-located.

---

## Overall biological conclusion

The sample is a **majority-endothelial vascular bed** (~7.4k endothelial vs ~0.7k
fibroblast cells with reads) with a minor, discrete fibroblast population. Endothelial cells
are **not discrete cell types but a continuum** on a shared `CDH5` (VE-cadherin) backbone,
structured by a small number of overlapping programmes.

### Marker glossary (what the genes mean here)

| Gene | Role | Programme it marks |
|---|---|---|
| `CDH5` (VE-cadherin) | pan-endothelial junctional identity | backbone (all endothelium) |
| `KDR` (VEGFR2) | main pro-angiogenic VEGF receptor | **VEGF / angiogenic core** |
| `VEGFA` | pro-angiogenic ligand (auto/paracrine) | **VEGF / angiogenic core** |
| `ITGA6` | laminin integrin, migration / basement membrane | VEGF / angiogenic core |
| `DLL4` | Notch ligand, **tip-cell selection** | **tip / sprouting** |
| `FLT1` (VEGFR1) | VEGF decoy receptor, tip/stalk balance | **tip / sprouting** |
| `MMP1` | matrix metalloprotease, ECM breakdown for sprouting | **tip / sprouting** |
| `SEMA3F` | semaphorin guidance cue | tip / sprouting |
| `ICAM1` | endothelial activation / adhesion | activated / Notch |
| `APLN` (apelin) | classic **tip-cell** marker | remodelling / apelin |
| `PDGFB` | pericyte-recruitment signal from tip cells | mural / pericyte |
| `ANGPT2` | Tie2 ligand, vessel destabilisation / remodelling | remodelling / apelin |
| `VWF`, `PECAM1` | mature/large-vessel endothelial markers | mature vessel |
| `JAG1` | Notch ligand (stalk / mural) | activated / Notch |
| `PDGFRB`, `COL1A1/2` | mural / fibroblast markers | non-angiogenic stroma |

### The endothelial gene-state clusters (Section 2, markers `CDH5`/`VWF`/`PECAM1` excluded from the clustering)

| State | n (%) | Defining markers (upregulated) | Spatial cohesion z | What it is | Angiogenic? |
|---|---|---|---|---|---|
| **c3 - quiescent backbone** | 4,589 (62%) | `CDH5` only, low `KDR` | +4.5 (mostly diffuse) | resting / phalanx endothelium | **no** |
| **c0 - VEGF-activated** | 1,383 (19%) | `KDR` + `VEGFA` | +6.6 | canonical angiogenic activation | **YES (core)** |
| **c2 - tip / sprouting** | 1,092 (15%) | `FLT1` + `DLL4` (+`VEGFA`) | **+9.3 (strongest patches)** | sprouting tip / stalk front | **YES (tip)** |
| **c1 - apelin / PDGFB sprout** | 324 (4%) | `APLN` + `PDGFB` (+`VEGFA`/`DLL4`) | +5.9 | apelin+ tip-like, pericyte-recruiting | **YES (tip-associated)** |

So the **angiogenic states are c0, c2 and c1**:
- **c0** is the *core VEGF-driven angiogenic* state - `KDR`(VEGFR2) receiving `VEGFA` on the
  `CDH5` backbone. This is the main axis of the whole endothelial continuum (it is also
  logistic-PCA axis 1: `KDR`+`VEGFA` vs the plain `CDH5` core).
- **c2** is the *tip / sprouting* state - `DLL4`+`FLT1`(+`MMP1`), i.e. Notch-driven tip
  selection plus ECM-remodelling for sprout invasion. It forms the **tightest spatial
  patches (z = +9.3)**, consistent with sprouting fronts being physically clustered.
- **c1** is a smaller *apelin/PDGFB* sprouting-associated state (`APLN` is a textbook
  tip-cell marker; `PDGFB` recruits pericytes) - a tip-to-mural transition flavour.
- **c3** (the 62% majority) is **not** angiogenic - it is `CDH5`-only quiescent endothelium,
  and it is spatially **diffuse** (z only +4.5), i.e. a shared baseline rather than a
  discrete patchy state.

### The same angiogenic signal in the other two views

- **Gene modules (Section 4).** With markers and collagen dropped and edges weighted by log2 OR,
  the co-occurrence graph resolves into **five modules (Q~0.21)** at resolution 1.5: a **VEGF /
  angiogenic core** (`KDR`, `VEGFA`, `ITGA6`), a **tip / sprouting** module (`DLL4`, `MMP1`,
  `SEMA3F`, `FLT1` - the genes defining states c0 and c2), a **mural / pericyte** module
  (`PDGFB`, `PDGFRB`), a **remodelling / apelin** module (`ANGPT2`, `APLN`), and a small
  **activated / inflammatory-Notch** module (`ICAM1`, `JAG1`). The two angiogenic modules and the
  mural vs remodelling split all *corroborate* the state/niche programmes; only `ICAM1`/`JAG1` is soft.
- **Spatial niches (Section 3).** The angiogenic niches are **niche 1** (`MMP1`/`VEGFA`/
  `DLL4` - a proteolytic-angiogenic front), **niche 3** (`FLT1`/`ITGA6`/`DLL4` - the
  tip/sprouting niche), and **niche 5** (`ICAM1`/`SEMA3F`/`PDGFB` - activated/guidance),
  with **niche 4** (`APLN`/`ANGPT2`) a remodelling flavour. Niche 2 (`PDGFRB`/`COL1A1`/
  `JAG1`) is mural/fibroblast and niche 0 is low-signal quiescent - the non-angiogenic
  environments.

**Bottom line.** The `KDR`/`VEGFA` **angiogenic activation** axis (state c0) and the
`DLL4`/`FLT1`/`MMP1` **tip/sprouting** axis (state c2) are the two angiogenic programmes, and
they recur as gene-state clusters, gene modules, and spatial niches - and form real spatial
patches. The bulk `CDH5`-only endothelium is quiescent and diffuse, and fibroblasts
(`PDGFRB`/collagen) are few, discrete, and spatially segregated from the angiogenic
endothelium.

**Caveats to keep in mind:** the gene-module partition is only **modestly modular** (Q~0.21,
and the module count is weight/resolution-dependent: phi gives 4, log2 OR splits the perivascular
block into mural + remodelling for 5) - the two angiogenic modules are robust and the mural vs
remodelling split is spatially supported, but the `ICAM1`/`JAG1` boundary should not be over-read;
all-0-read cells were assigned lineages artificially and carry no state; co-occurrence and spatial
adjacency are associations, not proof of interaction or regulation; very small clusters (e.g. c1
n=324, or the n~12 fibroblast state) and the chosen k / niche-count are not robust; marker
identities above are the conventional vascular-biology readings of these genes, not validated here;
and this is a single replicate.


# Step-by-step results write-up

A narrative walk-through of the key results with the figures embedded. **Run the code cell
directly below first** - it regenerates the key figures from the analysis above and saves
them to `report_figures/`, which the write-up then embeds.

In [4]:
# ==========================================================================
# Build the report figures for the write-up below, entirely from the saved part 2-4
# outputs loaded above, and write them to report_figures/ (embedded by the markdown).
# ==========================================================================

# 0) PART 2 - pairwise gene co-occurrence (all cells): log2 OR of significant pairs.
Mpair = pd.DataFrame(np.nan, index=GENES, columns=GENES)
for r in pair_sig[pair_sig["FDR"] < 0.05].itertuples():
    Mpair.loc[r.gene1, r.gene2] = r.log2_OR
    Mpair.loc[r.gene2, r.gene1] = r.log2_OR
lim = np.nanmax(np.abs(Mpair.values)) if np.isfinite(Mpair.values).any() else 1.0
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(Mpair, cmap="coolwarm", center=0, vmin=-lim, vmax=lim, square=True,
            mask=Mpair.isna(), ax=ax, cbar_kws={"label": "log2 odds ratio"})
ax.set_title("Pairwise gene co-occurrence (all cells, Fisher's exact FDR<0.05)")
plt.tight_layout(); plt.savefig(fig_dir / "fig0_pairwise.png", dpi=110, bbox_inches="tight"); plt.close()

# 1) PART 3 - endothelial continuum: confidence hist + logistic-PCA curve + latent scatter.
endo = cells[(cells["cell_type"] == "Endothelial") & (cells["bmm"] >= 0)]
uids = np.sort(endo["bmm"].unique()); K = len(uids)
id_to_idx = {b: i for i, b in enumerate(uids)}
mr = endo["bmm_confidence"].to_numpy()
lab = endo["bmm"].map(id_to_idx).to_numpy()
sc = endo[["lpca1", "lpca2"]].to_numpy()
fig, ax = plt.subplots(1, 3, figsize=(18, 4.5))
ax[0].hist(mr, bins=30, range=(1 / K, 1), color="steelblue"); ax[0].axvline(0.8, color="red", ls="--")
ax[0].set(title=f"Endothelial assignment confidence\n({100 * np.mean(mr >= 0.8):.0f}% cells >=0.8 -> continuum)",
          xlabel="max responsibility per cell", ylabel="cells")
ax[1].plot(endo_dev.index, 100 * endo_dev.values, "o-"); ax[1].grid(alpha=.3)
ax[1].set(title="Logistic PCA: no early plateau -> many small axes", ylim=(0, 100),
          xlabel="# latent axes", ylabel="% deviance explained")
cmap = plt.get_cmap("tab10", K); norm = BoundaryNorm(np.arange(-.5, K + .5, 1), K)
s = ax[2].scatter(sc[:, 0], sc[:, 1], c=lab, cmap=cmap, norm=norm, s=8, alpha=.6)
ax[2].set(title="Cells on latent axes (colour = Bernoulli state)", xlabel="axis 1", ylabel="axis 2")
plt.colorbar(s, ax=ax[2], ticks=range(K), label="state")
plt.tight_layout(); plt.savefig(fig_dir / "fig1_continuum.png", dpi=110, bbox_inches="tight"); plt.close()

# 2) PART 3 - gene-state cluster spatial co-location (within-image permutation z).
fig, ax = plt.subplots(figsize=(9, 8))
sns.heatmap(z_all, cmap="coolwarm", center=0, annot=True, fmt=".1f", square=True, ax=ax,
            cbar_kws={"label": "neighbour-enrichment z"})
ax.set_title("Do gene-state clusters co-locate? (within-image permutation z)")
plt.tight_layout(); plt.savefig(fig_dir / "fig2_spatial.png", dpi=110, bbox_inches="tight"); plt.close()

# 3) PART 4 - niche molecular fingerprint + cell-state composition.
fig, ax = plt.subplots(1, 2, figsize=(20, 5), gridspec_kw={"width_ratios": [2, 1.3]})
sns.heatmap(niche_gene_z.T, cmap="vlag", center=0, ax=ax[0], cbar_kws={"label": "z neighbourhood expr"})
ax[0].set_title("Niche molecular fingerprint"); ax[0].set_xlabel("niche")
sns.heatmap(state_comp, cmap="magma", annot=True, fmt=".2f", ax=ax[1], cbar_kws={"label": "fraction"})
ax[1].set_title("Cell-state composition of each niche"); ax[1].set_xlabel("state"); ax[1].set_ylabel("niche")
plt.tight_layout(); plt.savefig(fig_dir / "fig3_niches.png", dpi=110, bbox_inches="tight"); plt.close()

# 4) PART 4 - gene-interaction module graph (edge = log2 OR; markers excluded).
Gg = nx.Graph(); Gg.add_nodes_from(gene_modules["gene"])
for r in module_edges.itertuples():
    Gg.add_edge(r.g1, r.g2, weight=r.log2_OR)
module_of = dict(zip(gene_modules["gene"], gene_modules["module"]))
on_freq = dict(zip(gene_modules["gene"], gene_modules["on_freq"]))
conn = [g for g in Gg.nodes() if Gg.degree(g) > 0]
Hs = Gg.subgraph(conn)
lay = nx.spring_layout(Hs, weight="weight", k=1.4, seed=1)
n_mod = gene_modules["module"].nunique(); mcmap = plt.get_cmap("tab10", max(n_mod, 1))
maxf = max(on_freq.values())
nsz = [200 + 1600 * on_freq[g] / maxf for g in Hs.nodes()]
ncol = [mcmap(module_of[g]) for g in Hs.nodes()]
fig, ax = plt.subplots(figsize=(10, 8))
nx.draw_networkx_edges(Hs, lay, ax=ax, alpha=.35)
nx.draw_networkx_nodes(Hs, lay, ax=ax, node_size=nsz, node_color=ncol)
nx.draw_networkx_labels(Hs, lay, ax=ax, font_size=9)
ax.set_title("Endothelial gene-interaction modules (positive co-occurrence)"); ax.axis("off")
plt.tight_layout(); plt.savefig(fig_dir / "fig4_modules.png", dpi=110, bbox_inches="tight"); plt.close()

# 5) PART 4 - gene-module activity per niche.
fig, ax = plt.subplots(figsize=(1.4 * niche_module.shape[1] + 4, 0.5 * len(niche_module) + 3))
sns.heatmap(niche_module, cmap="viridis", annot=True, fmt=".2f", ax=ax,
            cbar_kws={"label": "mean fraction of module genes ON"})
ax.set_title("Gene-module activity across niches"); ax.set_xlabel("module"); ax.set_ylabel("niche")
plt.tight_layout(); plt.savefig(fig_dir / "fig5_module_niche.png", dpi=110, bbox_inches="tight"); plt.close()

print("Saved report figures ->", fig_dir)


Saved report figures -> report_figures


## 0. Which genes switch on together? (pairwise co-occurrence - part 2)

Before any clustering, part 2 asks the simplest question: across *all* cells, which pairs of
genes are switched on together more (or less) than chance? Each pair gets a 2x2 on/off table,
an odds ratio, and a Fisher's exact test (Benjamini-Hochberg FDR).

![Pairwise gene co-occurrence](report_figures/fig0_pairwise.png)

- **Warm (positive log2 OR)** pairs co-occur; **cool (negative)** pairs are mutually exclusive.
- The endothelial programme genes (`KDR`/`VEGFA`, `DLL4`/`FLT1`/`MMP1`) sit in warm blocks -
  they fire together - while the mural / collagen genes (`PDGFRB`/`COL1A1`/`COL1A2`) are
  **anti-correlated** with the endothelial markers (`CDH5` vs collagen is strongly negative).
  This is the first, assumption-free hint of the endothelial-vs-stroma split and the angiogenic
  sub-programmes that the later methods formalise.

## 1. Are endothelial cells discrete types or a continuum? (part 3)

Endothelial cells were clustered on their 18-gene on/off profile, with the lineage markers
`CDH5`/`VWF`/`PECAM1` **excluded** so the clusters capture *function*, not identity. We then
probed the result three independent ways.

![Endothelial continuum diagnostics](report_figures/fig1_continuum.png)

- **Left - assignment confidence.** Only ~38% of cells are confidently placed in one state
  (>=0.8); most sit *between* states. A discrete population would pile up against 1.0.
- **Middle - logistic PCA.** The deviance-explained curve keeps climbing with no early
  plateau - no single "dial" captures the cells; they spread along many small axes.
- **Right - cells on the latent axes.** One continuous cloud that the Bernoulli state colours
  merely slice up, rather than separated islands.

**Conclusion:** endothelial cells form a **structured continuum**, whose dominant axis is
`KDR`/`VEGFA` angiogenic activation on the shared `CDH5` backbone. (Run separately,
fibroblasts were 99% confidently sorted into 2 clean states - genuinely **discrete** - so the
continuum is specific to endothelium.)

## 2. Are the gene-state clusters "real" in tissue? (part 3)

Every cell was labelled with its state, then a within-image label-shuffling permutation test
asked whether cells of the same state sit together more than chance (diagonal) and which
states neighbour which (off-diagonal).

![Gene-state cluster spatial co-location](report_figures/fig2_spatial.png)

- The **angiogenic states are spatially cohesive** - the `KDR`/`VEGFA` state (z ~ +6.6) and
  especially the `FLT1`/`DLL4` tip state (z ~ +9.3, the tightest patches).
- The **`CDH5`-only backbone is spatially diffuse** (z ~ +4.5) - a shared baseline, not a
  discrete patch.
- Collagen fibroblasts sit **apart** from the angiogenic endothelium (negative off-diagonal),
  not intermixed.

**Conclusion:** the *functional* (angiogenic / sprouting) states occupy genuine tissue
territory; the backbone does not.

## 3. What defines the spatial niches? (part 4)

Niches are recurring neighbourhood micro-environments, built from the neighbourhood
expression of the informative genes only (cell-type composition, entropy and lineage markers
were deliberately excluded, or the niches just track "endothelial-dense vs mixed").

![Niche molecular fingerprint and cell-state composition](report_figures/fig3_niches.png)

Each niche has a distinct molecular signature (left) and cell-state composition (right):
a `MMP1`/`VEGFA` proteolytic-angiogenic niche, a `FLT1`/`DLL4`/`ITGA6` tip/sprouting niche,
an `APLN`/`ANGPT2` remodelling niche, and a `PDGFRB`/`JAG1` mural/fibroblast niche.

## 4. Which genes act together? (gene modules - part 4)

A gene-gene graph of significant **positive** co-occurrences in endothelial cells, with edge
weight = **log2 odds ratio** (effect size). The pan-endothelial markers `CDH5`/`VWF`/`PECAM1` and
collagen are **excluded** (ON almost everywhere / stromal, so they carry no module structure);
communities on the remaining functional genes are co-firing **modules** (resolution 1.5).

![Gene-interaction modules](report_figures/fig4_modules.png)

Five modules emerge: a **VEGF / angiogenic core** (`KDR`, `VEGFA`, `ITGA6`), a **tip/sprouting**
module (`DLL4`, `MMP1`, `FLT1`, `SEMA3F`), a **mural / pericyte** module (`PDGFB`, `PDGFRB`), a
**remodelling / apelin** module (`ANGPT2`, `APLN`), and a small **activated / inflammatory-Notch**
module (`ICAM1`, `JAG1`). The two angiogenic modules are stable across metrics; the log2-OR
weighting additionally separates the mural (`PDGFB`/`PDGFRB`) and remodelling (`ANGPT2`/`APLN`)
genes that phi keeps in one perivascular block.

## 5. Do the modules map onto tissue? (module x niche - part 4)

![Gene-module activity across niches](report_figures/fig5_module_niche.png)

The **VEGF-core and tip/sprouting modules light up in exactly the angiogenic / tip niches**,
tying those programmes to physical tissue. The **mural** (`PDGFB`/`PDGFRB`) and **remodelling**
(`ANGPT2`/`APLN`) modules now map to their own niches (the `PDGFRB` mural niche and the
`APLN`/`ANGPT2` niche) - so the 5-module split is spatially corroborated. Only the `ICAM1`/`JAG1`
module stays split across niches.

## How the methods agree

Four independent views - pairwise co-occurrence (part 2), clustering the cells (part 3), a
gene co-occurrence graph and spatial neighbourhoods (part 4) - converge on the **same 2-3
endothelial programmes**:

| Programme | Pairwise (Sec 0) | Gene-state cluster (Sec 1) | Gene module (Sec 4) | Spatial niche (Sec 3) |
|---|---|---|---|---|
| **VEGF / angiogenic core** | `KDR`-`VEGFA` warm block | c0 `KDR`/`VEGFA` | VEGF-core module (`KDR`/`VEGFA`/`ITGA6`) | niche 1 & 3 (`MMP1`/`VEGFA`, `FLT1`/`DLL4`) |
| **Tip / sprouting** | `DLL4`-`MMP1`-`FLT1` warm | c2 `FLT1`/`DLL4` | tip/sprouting module (`DLL4`/`MMP1`/`SEMA3F`/`FLT1`) | niche 3 (`FLT1`/`DLL4`) |
| **Mural / pericyte** | collagen block, anti-endothelial | c1 `APLN`/`PDGFB` | mural module (`PDGFB`/`PDGFRB`) | niche 2 (`PDGFRB`) |
| **Remodelling / apelin** | `APLN`-`ANGPT2` warm | c1 `APLN`/`PDGFB` | remodelling module (`ANGPT2`/`APLN`) | niche 4 (`APLN`/`ANGPT2`) |
| **Activated / Notch** | - | (within activated endothelium) | Notch module (`ICAM1`/`JAG1`) | niche 5 & 2 (`ICAM1`; `JAG1`) |
| **Quiescent backbone** | `CDH5` ubiquitous | c3 `CDH5` only | (no module - markers excluded) | niche 0 (low signal) |

The two angiogenic programmes (core VEGF activation and tip/sprouting) additionally form
**genuine spatial patches**, while the quiescent `CDH5` backbone is spread diffusely. See the

**Overall biological conclusion** above for the full marker glossary and caveats.




**Overall biological conclusion** above for the full marker glossary and caveats.